In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
os.chdir('/content/drive/MyDrive/UNSW_Blanced')

In [3]:
!pip install adversarial-robustness-toolbox


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 69.7 MB/s eta 0:00:00


In [4]:
#@title Env & setup
!pip -q install numpy pandas scikit-learn scipy

import os, time, random, math, json
import numpy as np
import pandas as pd
from math import ceil
from typing import Dict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score

from scipy.stats import norm
from scipy.stats import beta as beta_dist

# ------------------ Repro / Device ------------------
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


device(type='cuda')

In [5]:


#@title Load UNSW dataset → tensors
import pandas as pd
import numpy as np
import torch

train = pd.read_csv('unsw_multi_train_processed.csv')
test = pd.read_csv('unsw_multi_test_processed.csv')

X_train = train.drop(columns=['Label']).values
y_train = train['Label'].values
X_test  = test.drop(columns=['Label']).values
y_test  = test['Label'].values

X_train_tensor = torch.tensor(np.array(X_train), dtype=torch.float32)
y_train_tensor = torch.tensor(np.array(y_train), dtype=torch.long)
X_test_tensor  = torch.tensor(np.array(X_test),  dtype=torch.float32)
y_test_tensor  = torch.tensor(np.array(y_test),  dtype=torch.long)

print("Train:", X_train_tensor.shape, y_train_tensor.shape)
print("Test :", X_test_tensor.shape,  y_test_tensor.shape)




Train: torch.Size([175341, 194]) torch.Size([175341])
Test : torch.Size([82332, 194]) torch.Size([82332])


In [6]:
#@title Base model
import torch.nn as nn

class TabularNN(nn.Module):
    def __init__(self, input_size, num_classes, dropout_rate=0.2):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, num_classes)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)
        x = torch.relu(self.fc3(x))
        return self.fc4(x)


In [7]:
num_classes = 10
model = TabularNN(input_size=X_train_tensor.shape[1], num_classes=num_classes).to(device)

In [8]:
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test, dtype=torch.long)

In [ ]:
#@title Train base model (clean)
input_size = X_train_tensor.shape[1]
num_classes = len(torch.unique(y_train_tensor))

model = TabularNN(input_size=input_size, num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_tensor,  y_test_tensor),  batch_size=256, shuffle=False)

results_file = "results_baseline/url_evaluation_results.txt"
os.makedirs(os.path.dirname(results_file), exist_ok=True)

with open(results_file, "w") as f:
    f.write("Model Evaluation Results\n" + "="*50 + "\n")

epochs = 100
for epoch in range(epochs):
    model.train()
    run = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        run += float(loss.item())

    ep_loss = run / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] Loss: {ep_loss:.4f}")
    with open(results_file, "a") as f:
        f.write(f"Epoch [{epoch+1}/{epochs}] Loss: {ep_loss:.4f}\n")

# Clean test accuracy
model.eval()
with torch.no_grad():
    logits = model(X_test_tensor.to(device))
    pred = torch.argmax(logits, dim=1).cpu()

clean_acc = accuracy_score(y_test_tensor.cpu().numpy(), pred.numpy())
print(f"Clean test accuracy: {clean_acc:.4f}")

with open(results_file, "a") as f:
    f.write(f"\nClean test accuracy: {clean_acc:.4f}\n")

Epoch [1/100] Loss: 0.6431
Epoch [2/100] Loss: 0.5576
Epoch [3/100] Loss: 0.5383
Epoch [4/100] Loss: 0.5235
Epoch [5/100] Loss: 0.5140
Epoch [6/100] Loss: 0.5085
Epoch [7/100] Loss: 0.5044
Epoch [8/100] Loss: 0.5007
Epoch [9/100] Loss: 0.4982
Epoch [10/100] Loss: 0.4964
Epoch [11/100] Loss: 0.4934
Epoch [12/100] Loss: 0.4914
Epoch [13/100] Loss: 0.4888
Epoch [14/100] Loss: 0.4871
Epoch [15/100] Loss: 0.4852
Epoch [16/100] Loss: 0.4834
Epoch [17/100] Loss: 0.4809
Epoch [18/100] Loss: 0.4801
Epoch [19/100] Loss: 0.4796
Epoch [20/100] Loss: 0.4779
Epoch [21/100] Loss: 0.4752
Epoch [22/100] Loss: 0.4756
Epoch [23/100] Loss: 0.4753
Epoch [24/100] Loss: 0.4732
Epoch [25/100] Loss: 0.4733
Epoch [26/100] Loss: 0.4726
Epoch [27/100] Loss: 0.4717
Epoch [28/100] Loss: 0.4697
Epoch [29/100] Loss: 0.4699
Epoch [30/100] Loss: 0.4696
Epoch [31/100] Loss: 0.4687
Epoch [32/100] Loss: 0.4677
Epoch [33/100] Loss: 0.4671
Epoch [34/100] Loss: 0.4661
Epoch [35/100] Loss: 0.4657
Epoch [36/100] Loss: 0.4653
E

In [ ]:
#@title Evaluate base model on adversarial matrices (pre-defense)
def robust_load_matrix(path: str) -> np.ndarray:
    try:
        return np.loadtxt(path, delimiter=",", dtype=np.float32)
    except Exception:
        return np.loadtxt(path, dtype=np.float32)

# Example loaders (URL). Adjust paths if needed.
adversarial_loaders = {
    "PGD": "X_adv_pgd_unsw2_brmulti.txt",
    "JSMA": "X_adv_jsma_unsw_multi.txt",
    "ZOO": "X_adv_zoo_agg_unsw_brmulti.txt",
    "DeepFool": "X_adv_pgd_unsw2_brmulti.txt",
    "FGSM": "X_adv_fgsm_unsw2_brmulti.txt",
    #"CW": "X_adv_cw_agg_unsw.txt",
    #"LPF": "X_adv_lpf_unsw.csv",
    #"HSJ": "X_adv_hsj_unsw.txt",
}

for attack_name, file_path in adversarial_loaders.items():
    X_adv = robust_load_matrix(file_path)

    if X_adv.shape[1] != X_train_tensor.shape[1]:
        print(f"[SKIP] {attack_name}: {X_adv.shape[1]} features != {X_train_tensor.shape[1]}")
        continue

    X_adv_tensor = torch.tensor(X_adv, dtype=torch.float32).to(device)
    t0 = time.time()
    with torch.no_grad():
        adv_pred = model(X_adv_tensor).argmax(1).cpu().numpy()
    dt = time.time() - t0

    adv_acc = accuracy_score(y_test_tensor.numpy(), adv_pred)
    print(f"{attack_name} adversarial accuracy: {adv_acc:.4f} | time {dt:.2f}s")
    with open(results_file, "a") as f:
        f.write(f"{attack_name} adversarial accuracy: {adv_acc:.4f}\n")
        f.write(f"Time taken for {attack_name}: {dt:.2f}s\n")


In [19]:
import numpy as np
import torch
from scipy.stats import norm, beta as beta_dist

class Smooth:
    """
    Original Randomized Smoothing certification wrapper.

    - Train base classifier on clean data for ORS-Z
    - Use fixed Gaussian noise only at certification time
    - Supports both binary and multi-class certification
    """

    ABSTAIN = -1

    def __init__(self, base_classifier, num_classes: int, sigma: float, device=None):
        self.base_classifier = base_classifier
        self.num_classes = int(num_classes)
        self.sigma = float(sigma)
        if device is None:
            device = next(base_classifier.parameters()).device
        self.device = device

    @staticmethod
    def _cp_lower(k: int, n: int, alpha: float) -> float:
        if n <= 0 or k <= 0:
            return 0.0
        return float(beta_dist.ppf(alpha, k, n - k + 1))

    @staticmethod
    def _cp_upper(k: int, n: int, alpha: float) -> float:
        if n <= 0 or k >= n:
            return 1.0
        return float(beta_dist.ppf(1.0 - alpha, k + 1, n - k))

    @staticmethod
    def _ppf_safe(p: float) -> float:
        p = float(np.clip(p, 1e-12, 1 - 1e-12))
        return float(norm.ppf(p))

    @torch.no_grad()
    def _sample_noise(self, x: torch.Tensor, num: int, batch_size: int) -> np.ndarray:
        self.base_classifier.eval()
        counts = np.zeros(self.num_classes, dtype=np.int64)

        remaining = int(num)
        while remaining > 0:
            bs = min(batch_size, remaining)
            remaining -= bs

            batch = x.repeat((bs, 1)).to(self.device).float()
            noise = torch.randn_like(batch) * self.sigma

            logits = self.base_classifier(batch + noise)
            preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
            counts += np.bincount(preds, minlength=self.num_classes)

        return counts

    def certify(self, x: torch.Tensor, n0: int, n: int, alpha: float, batch_size: int):
        x = x.to(self.device).float()
        self.base_classifier.eval()

        # 1) class selection
        counts_selection = self._sample_noise(x, n0, batch_size)
        cA_hat = int(np.argmax(counts_selection))

        # 2) probability estimation
        counts_estimation = self._sample_noise(x, n, batch_size)
        nA = int(counts_estimation[cA_hat])
        pA_LB = self._cp_lower(nA, n, alpha)

        # binary case
        if self.num_classes == 2:
            if pA_LB <= 0.5:
                return Smooth.ABSTAIN, 0.0
            R = self.sigma * self._ppf_safe(pA_LB)
            return cA_hat, max(0.0, float(R))

        # multi-class case: use runner-up upper bound
        cB_hat = int(np.argsort(counts_estimation)[-2])
        nB = int(counts_estimation[cB_hat])
        pB_UB = self._cp_upper(nB, n, alpha)

        if pA_LB <= 0.5 or pA_LB <= pB_UB:
            return Smooth.ABSTAIN, 0.0

        zA = self._ppf_safe(pA_LB)
        zB = self._ppf_safe(pB_UB)
        R = 0.5 * self.sigma * (zA - zB)

        return cA_hat, max(0.0, float(R))

    def set_sigma(self, sigma: float):
        self.sigma = float(sigma)

In [20]:
import os
import time
import numpy as np
import torch

def test_on_adversarial_original(
    smoother,
    X_adv_tensor,
    y_true_np,
    n0=200,
    n=1000,
    alpha=0.001,
    batch_size=64,
    save_to: str = None,
    tag: str = "",
):
    """
    Evaluate Original Randomized Smoothing (ORS) on an adversarial matrix.

    Metrics:
      - Acc: base classifier accuracy on adversarial samples (NO noise)
      - Certified Acc: fraction where certify() returns correct class (and radius>0 implicitly)
      - Abstain: fraction where certify() returns ABSTAIN
      - Avg R: mean certified radius over *certified* samples only (radius>0)
      - TimeTot: total wall-clock time over all samples
      - Cert ms/sample: (certify_time / N) * 1000

    Args:
      smoother: instance of Smooth (ORS certifier), has base_classifier and certify()
      X_adv_tensor: torch.Tensor [N, d] (on any device; will be moved to smoother.device)
      y_true_np: numpy array [N] or list/torch of labels aligned with X_adv_tensor rows
    Returns:
      (accuracy_pct, certified_accuracy_pct, abstention_rate_pct,
       avg_certified_radius, total_time_seconds, certify_ms_per_sample)
    """
    device = getattr(smoother, "device", None)
    if device is None:
        device = next(smoother.base_classifier.parameters()).device

    base_model = smoother.base_classifier
    base_model.eval()


    if torch.is_tensor(y_true_np):
        y = y_true_np.detach().cpu().numpy().astype(int)
    else:
        y = np.asarray(y_true_np, dtype=int)

    X_adv = X_adv_tensor.to(device).float()
    total = int(X_adv.shape[0])

    correct = 0
    certified_correct = 0
    abstain = 0
    radii = []

    t_all_start = time.time()
    certify_time = 0.0

    with torch.no_grad():
        for i in range(total):
            x_i = X_adv[i].unsqueeze(0)
            y_i = int(y[i])

            # --- base accuracy (no smoothing noise) ---
            pred = int(torch.argmax(base_model(x_i), dim=1).item())
            if pred == y_i:
                correct += 1

            # --- certification (smoothing noise inside certify) ---
            t0 = time.time()
            c_hat, r_hat = smoother.certify(x_i, n0=n0, n=n, alpha=alpha, batch_size=batch_size)
            certify_time += (time.time() - t0)

            if c_hat == smoother.ABSTAIN:
                abstain += 1
            else:
                # certified correctness: predicted class equals true label AND radius>0
                if int(c_hat) == y_i and float(r_hat) > 0:
                    certified_correct += 1
                    radii.append(float(r_hat))

    total_time = time.time() - t_all_start
    ms_per_sample = (certify_time / max(total, 1)) * 1000.0

    acc_pct = 100.0 * correct / max(total, 1)
    cert_acc_pct = 100.0 * certified_correct / max(total, 1)
    abstain_pct = 100.0 * abstain / max(total, 1)
    avg_r = float(np.mean(radii)) if len(radii) else 0.0

    msg = (f"{tag} ORS | Acc: {acc_pct:.2f}% | Certified: {cert_acc_pct:.2f}% | "
           f"Abstain: {abstain_pct:.2f}% | Avg R: {avg_r:.4f} | "
           f"TimeTot: {total_time:.2f}s | Cert ms/sample: {ms_per_sample:.2f}")
    print(msg)

    if save_to is not None:
        d = os.path.dirname(save_to)
        if d:
            os.makedirs(d, exist_ok=True)
        with open(save_to, "a") as f:
            f.write(msg + "\n")

    return acc_pct, cert_acc_pct, abstain_pct, avg_r, total_time, ms_per_sample


In [21]:
adversarial_loaders = {
   # "PGD": "X_adv_pgd_unsw2_brmulti.txt",
    #"JSMA": "X_adv_jsma_unsw_multi.txt",
    #"ZOO": "X_adv_zoo_agg_unsw_brmulti.txt",
    "DeepFool": "X_adv_pgd_unsw2_brmulti.txt",
    "FGSM": "X_adv_fgsm_unsw2_brmulti.txt",
    #"CW": "X_adv_cw_agg_unsw.txt",
    #"LPF": "X_adv_lpf_unsw.csv",
    #"HSJ": "X_adv_hsj_unsw.txt",
}

In [22]:
import numpy as np

def load_adv_matrix(path: str) -> np.ndarray:
    try:
        return np.loadtxt(path, delimiter=",", dtype=np.float32)
    except Exception:
        return np.loadtxt(path, dtype=np.float32)


In [23]:
import os
import torch

# ---- ORS config ----
sigma_cert = 0.3
n0 = 200
n = 1000
alpha = 0.001
batch_size_cert = 64

num_classes = int(max(y_train_tensor.max(), y_test_tensor.max()).item()) + 1

# Create ORS smoother using your trained base model
smoother = Smooth(
    base_classifier=model,
    num_classes=num_classes,
    sigma=sigma_cert,
    device=device
)

# Output log
ors_log = "results_baseline/ORS_eval_sigma0.30.txt"
os.makedirs(os.path.dirname(ors_log), exist_ok=True)
with open(ors_log, "w") as f:
    f.write(f"===== ORS Evaluation (sigma={sigma_cert}) =====\n")
    f.write(f"num_classes={num_classes}, n0={n0}, n={n}, alpha={alpha}, batch_size={batch_size_cert}\n\n")

# Evaluate clean accuracy (BASE) for reference
model.eval()
with torch.no_grad():
    pred_clean = model(X_test_tensor.to(device)).argmax(dim=1).cpu()
clean_acc = (pred_clean.numpy() == y_test_tensor.cpu().numpy()).mean() * 100.0

print(f"[BASE] Clean test acc: {clean_acc:.2f}%")
with open(ors_log, "a") as f:
    f.write(f"[BASE] Clean test acc: {clean_acc:.2f}%\n\n")

# Now evaluate ORS on each adversarial set
for attack, fp in adversarial_loaders.items():
    mat = load_adv_matrix(fp)


    if mat.shape != tuple(X_test_tensor.shape):
        raise ValueError(f"{attack} shape {mat.shape} != X_test {tuple(X_test_tensor.shape)}")

    X_adv_tensor = torch.tensor(mat, dtype=torch.float32, device=device)

    acc, cert_acc, abstain_pct, avg_R, time_s, ms_ps = test_on_adversarial_original(
        smoother=smoother,
        X_adv_tensor=X_adv_tensor,
        y_true_np=y_test_tensor.cpu().numpy(),
        n0=n0,
        n=n,
        alpha=alpha,
        batch_size=batch_size_cert,
        save_to=ors_log,
        tag=f"[ORS sigma={sigma_cert}] [attack={attack}]",
    )

print("Saved ORS evaluation log to:", ors_log)

[BASE] Clean test acc: 76.30%
[ORS sigma=0.3] [attack=PGD] ORS | Acc: 33.87% | Certified: 12.92% | Abstain: 69.02% | Avg R: 0.1345 | TimeTot: 823.11s | Cert ms/sample: 9.57
[ORS sigma=0.3] [attack=JSMA] ORS | Acc: 33.45% | Certified: 7.54% | Abstain: 70.99% | Avg R: 0.3403 | TimeTot: 825.87s | Cert ms/sample: 9.60
[ORS sigma=0.3] [attack=ZOO] ORS | Acc: 73.77% | Certified: 18.98% | Abstain: 80.57% | Avg R: 0.1228 | TimeTot: 823.24s | Cert ms/sample: 9.57
[ORS sigma=0.3] [attack=DeepFool] ORS | Acc: 33.87% | Certified: 12.81% | Abstain: 69.19% | Avg R: 0.1352 | TimeTot: 827.13s | Cert ms/sample: 9.62
[ORS sigma=0.3] [attack=FGSM] ORS | Acc: 34.87% | Certified: 13.72% | Abstain: 64.59% | Avg R: 0.1174 | TimeTot: 829.53s | Cert ms/sample: 9.64
Saved ORS evaluation log to: results_baseline/ORS_eval_sigma0.30.txt
